# Course: Data Science Course
# Author: Sandro Camargo <sandrocamargo@unipampa.edu.br>
# Class 99 - Prais-Winsten Regression Datasus


To open this code in your Google Colab environment, [click here](https://colab.research.google.com/github/Sandrocamargo/data-science/blob/master/cd99_RegressaoPraisWinsten.ipynb).

# Projeto 1

**Descrição geral**

Neste projeto, o estudante deverá realizar um estudo de análise temporal utilizando uma base de dados real, com o objetivo de investigar a evolução de um fenômeno ao longo do tempo.

A base de dados deve ser original e individual, não podendo coincidir com a de outros colegas.

**Contexto aplicado**

A Secretaria de Saúde do Rio Grande do Sul busca identificar quais categorias de doenças apresentam maior crescimento ao longo dos anos entre a população gaúcha.

Para isso, o estudo deverá utilizar dados oficiais do DATASUS.

**Objetivos**

O estudante deverá selecionar uma única categoria da Classificação Internacional de Doenças, 10ª (CID-10) e desenvolver as seguintes análises:

* Análise de tendência temporal (4 pontos)
* Análise comparativa normalizada (2 pontos)
* Comunicação dos resultados (4 pontos)
* Produção científica (1 ponto extra)

# Carga de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import linregress
import statsmodels.api as sm
import math
from statsmodels.regression.linear_model import yule_walker
from statsmodels.regression.linear_model import GLSAR

# Carga de dados obtidos a partir do portal DATASUS

* Morbidade Hospitalar do SUS - por local de residência - Rio Grande do Sul
* Linha: Por região e UF
* Coluna: Por ano de atendimento
* Conteúdo: Internações
* Internações por Ano atendimento segundo Região de Saúde (CIR)
* Capítulo CID-10: II. Neoplasias (tumores) ** ALTERE DE ACORDO COM SEU PROJETO **
* Lista Morb CID-10: Neoplasia maligna da pele ** NÃO É NECESSÁRIO INFORMAR **
* Período: Jan/2008-Dez/2025
* colunas separadas por ;


https://datasus.saude.gov.br/acesso-a-informacao/morbidade-hospitalar-do-sus-sih-sus/


In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/Sandrocamargo/data-science/refs/heads/main/datasets/melanoma-br-2008-2025.txt", sep=";", na_values="-")
df.info()

### Apresentando o boxplot dos totais por coluna

In [ ]:
# remover última linha que tem o somatório dos totais
total_sem_ultima = df["Total"].iloc[:-1]

plt.figure(figsize=(6, 4))
plt.boxplot(total_sem_ultima, tick_labels=["Total"])

plt.title("Boxplot da coluna Total (sem última linha)")
plt.ylabel("Casos")

plt.show()

### Mostrando os primeiros valores para uma inspeção visual. Verifique existência de valores NaN

In [ ]:
df.head(n=10)

### Verifique se alguma das colunas apresenta uma menor contagem de elementos. Isso indica valores NaN

In [ ]:
print(df.describe())

### Preparação dos dados. Eliminação de dados inconsistentes de 2007. Eliminação de colunas que tenham valores NA

In [ ]:
df.drop(columns=["2007"], inplace=True)
df.dropna(axis=1, inplace=True)

In [ ]:
df.head(n=10)

### Gera modelo de regressão dos valores totais

In [ ]:
x = df.columns.values[1:-1].astype(int)
y = df.iloc[-1, 1:-1].values.astype(int)

# variável temporal
x = np.arange(len(y))

# matriz de desenho
X = sm.add_constant(x)

# modelo Prais-Winsten aproximado via GLSAR(1)
modelo = GLSAR(y, X, rho=1)
resultado = modelo.iterative_fit(maxiter=10)

# coeficientes
b0 = resultado.params[0]
b1 = resultado.params[1]

# estatísticas
r2 = resultado.rsquared
p_value = resultado.pvalues[1]

print(resultado.summary())

In [ ]:
anos = [int(col) for col in df.columns if str(col).isdigit()]

# valores ajustados
y_pred = resultado.fittedvalues

plt.figure(figsize=(10,6))

plt.scatter(anos, y, label='Observado')
plt.plot(anos, y_pred, '--', color="red", linewidth=2,
         label='Prais-Winsten')

plt.xlabel('Ano')
plt.ylabel('Número de internações')
plt.title('Tendência temporal das internações por neoplasia maligna da pele')
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
y_log = np.log10(y)

X = sm.add_constant(x)

modelo = GLSAR(y_log, X, rho=1)
resultado = modelo.iterative_fit(maxiter=10)

beta = resultado.params[1]

# Annual Percent Change (APC)
apc = (10**beta - 1) * 100

print(f"APC = {apc:.2f}% ao ano")

In [ ]:
y_pred = 10**resultado.fittedvalues

plt.figure(figsize=(10,6))

plt.scatter(anos, y, label='Observado')
plt.plot(anos, y_pred, '--', color="red",
         linewidth=2,
         label='Prais-Winsten')

plt.xlabel('Ano')
plt.ylabel('Número de internações')
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
beta = resultado.params[1]
erro_padrao = resultado.bse[1]
p_valor = resultado.pvalues[1]
ic_inf = beta - 1.96 * erro_padrao
ic_sup = beta + 1.96 * erro_padrao

apc = (10**beta - 1) * 100
apc_inf = (10**ic_inf - 1) * 100
apc_sup = (10**ic_sup - 1) * 100

print(f"β = {beta:.5f}")
print(f"p = {p_valor:.4f}")
print(f"APC = {apc:.2f}%")
print(f"IC95% APC = {apc_inf:.2f}% a {apc_sup:.2f}%")

In [ ]:
from scipy.stats import t

pred = resultado.get_prediction(X)
pred_summary = pred.summary_frame()

plt.figure(figsize=(10,6))

plt.scatter(anos, y, label='Observado')

plt.plot(
    anos,
    10**pred_summary['mean'],
    '--',
    color = "red",
    linewidth=2,
    label='Tendência ajustada'
)

plt.fill_between(
    anos,
    10**pred_summary['mean_ci_lower'],
    10**pred_summary['mean_ci_upper'],
    alpha=0.1,
    color="red",
    label='IC95%'
)

plt.text(
    0.05, 0.95,
    f'APC = {apc:.2f}%\np = {p_valor:.3f}',
    transform=plt.gca().transAxes,
    verticalalignment='top'
)

plt.xlabel('Ano')
plt.ylabel('Número de internações')
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.regression.linear_model import GLSAR

# --- preparar dados ---
colunas_anos = [c for c in df.columns if c.isdigit()]
anos = np.array(colunas_anos, dtype=int)

# matriz de valores
Y = df[colunas_anos].astype(float).values

# variável tempo
tempo = np.arange(len(anos))

# matriz X
X = sm.add_constant(tempo)

# listas para armazenar resultados
betas = []
interceptos = []
p_values = []
r2s = []
apcs = []
apc_inf = []
apc_sup = []
medias = []

for y in Y:

    # evitar problemas com log(0)
    if np.any(y <= 0):
        betas.append(np.nan)
        interceptos.append(np.nan)
        p_values.append(np.nan)
        r2s.append(np.nan)
        apcs.append(np.nan)
        apc_inf.append(np.nan)
        apc_sup.append(np.nan)
        medias.append(np.nan)
        continue

    # transformação logarítmica
    y_log = np.log10(y)

    # Prais-Winsten (aproximação GLSAR)
    modelo = GLSAR(y_log, X, rho=1)
    resultado = modelo.iterative_fit(maxiter=10)

    beta = resultado.params[1]
    intercepto = resultado.params[0]

    erro = resultado.bse[1]

    # APC
    apc = (10**beta - 1) * 100

    # IC95%
    ic_inf_beta = beta - 1.96 * erro
    ic_sup_beta = beta + 1.96 * erro

    apc_ic_inf = (10**ic_inf_beta - 1) * 100
    apc_ic_sup = (10**ic_sup_beta - 1) * 100

    betas.append(beta)
    interceptos.append(intercepto)
    p_values.append(resultado.pvalues[1])

    # pseudo-R²
    r2s.append(np.corrcoef(y_log,
                           resultado.fittedvalues)[0,1]**2)

    apcs.append(apc)
    apc_inf.append(apc_ic_inf)
    apc_sup.append(apc_ic_sup)

    medias.append(np.mean(y))

# dataframe final
resultados_df = pd.DataFrame({
    "Estado": df.iloc[:,0],
    "Beta": betas,
    "Intercepto": interceptos,
    "p_valor": p_values,
    "R2": r2s,
    "Media": medias,
    "APC (%)": apcs,
    "IC95_inf": apc_inf,
    "IC95_sup": apc_sup
})

# classificação da tendência
resultados_df["Tendencia"] = np.where(
    (resultados_df["p_valor"] < 0.05) &
    (resultados_df["APC (%)"] > 0),
    "Crescente",
    np.where(
        (resultados_df["p_valor"] < 0.05) &
        (resultados_df["APC (%)"] < 0),
        "Decrescente",
        "Estacionária"
    )
)

resultados_df = resultados_df.sort_values(
    by="APC (%)",
    ascending=False
)

resultados_df.head(110)

### Mostrando o comportamento das regiões extremas: com maior crescimento e menor crescimento.

In [ ]:
# --- função para obter série ---
def get_series(regiao):
    return (
        df.loc[df.iloc[:, 0] == regiao, colunas_anos]
        .values.flatten()
        .astype(float)
    )

from statsmodels.regression.linear_model import GLSAR
import statsmodels.api as sm

# --- função para Prais-Winsten ---
def trend_pw(y):

    y = np.asarray(y, dtype=float)

    tempo = np.arange(len(y))
    y_log = np.log10(y)

    X = sm.add_constant(tempo)

    modelo = GLSAR(y_log, X, rho=1)
    resultado = modelo.iterative_fit(maxiter=10)

    # predição
    y_log_pred = resultado.fittedvalues

    # erro padrão das predições
    cov = resultado.cov_params()

    se_pred = np.sqrt(
        cov[0, 0]
        + (tempo**2) * cov[1, 1]
        + 2 * tempo * cov[0, 1]
    )

    # IC95% na escala log
    y_log_inf = y_log_pred - 1.96 * se_pred
    y_log_sup = y_log_pred + 1.96 * se_pred

    # volta para escala original
    y_pred = 10**y_log_pred
    y_inf = 10**y_log_inf
    y_sup = 10**y_log_sup

    beta = resultado.params[1]
    erro = resultado.bse[1]

    apc = (10**beta - 1) * 100

    ic_inf_apc = (10**(beta - 1.96 * erro) - 1) * 100
    ic_sup_apc = (10**(beta + 1.96 * erro) - 1) * 100

    p = resultado.pvalues[1]

    r2 = np.corrcoef(
        y_log,
        resultado.fittedvalues
    )[0, 1]**2

    return (
        y_pred,
        y_inf,
        y_sup,
        apc,
        ic_inf_apc,
        ic_sup_apc,
        r2,
        p
    )

def format_p(p):
    return "< 0.001" if p < 0.001 else f"= {p:.3f}"

# --- lista de regiões ---
regioes = (
    resultados_df[
        resultados_df["Estado"].str.startswith("Região")
    ]["Estado"]
    .tolist()
)

# --- layout ---
n = len(regioes)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = axes.reshape(nrows, ncols)

# --- loop ---
for i, regiao in enumerate(regioes):

    row = i // ncols
    col = i % ncols

    ax = axes[row, col]

    y = get_series(regiao)

    # Prais-Winsten
    (
      y_pred,
      y_inf,
      y_sup,
      apc,
      ic_inf,
      ic_sup,
      r2,
      p
    ) = trend_pw(y)

    x_vals = anos.flatten()

    # observados
    ax.plot(
        x_vals,
        y,
        marker='o',
        label='Observado'
    )

    # ajustados
    ax.plot(
        x_vals,
        y_pred,
        linestyle='--',
        linewidth=2,
        label='Prais-Winsten'
    )

    ax.fill_between(
      x_vals,
      y_inf,
      y_sup,
      alpha=0.1,
      color="red",
      label='IC95%'
    )

    ax.set_title(regiao)
    ax.set_xlabel("Ano")
    ax.set_ylabel("Casos")

    texto = (
        f"APC = {apc:.2f}%\n"
        f"IC95% = [{ic_inf:.2f}; {ic_sup:.2f}]\n"
        #f"R² = {r2:.2f}\n"
        f"p {format_p(p)}"
    )

    ax.text(
        0.05,
        0.95,
        texto,
        transform=ax.transAxes,
        fontsize=8,
        verticalalignment='top',
        bbox=dict(
            boxstyle="round",
            alpha=0.2
        )
    )

    ax.legend(fontsize=7)

# --- remover eixos vazios ---
for j in range(i + 1, nrows * ncols):
    fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------
# 1. calcular métricas
# ---------------------------

regioes = (
    resultados_df[
        ~resultados_df["Estado"].str.startswith("Região")
    ]["Estado"]
    .tolist()
)

resultados_lista = []

for regiao in regioes:

    y = get_series(regiao)

    (
      y_pred,
      y_inf,
      y_sup,
      apc,
      ic_inf,
      ic_sup,
      r2,
      p
    ) = trend_pw(y)

    resultados_lista.append({
        "Estado": regiao,
        "y": y,
        "y_pred": y_pred,
        "y_inf": y_inf,
        "y_sup": y_sup,
        "apc": apc,
        "ic_inf": ic_inf,
        "ic_sup": ic_sup,
        "p": p
    })

res_df = pd.DataFrame(resultados_lista)

# ---------------------------
# 2. filtrar significativos
# ---------------------------

res_sig = res_df[res_df["p"] < 0.05].copy()

# ---------------------------
# 3. selecionar extremos
# ---------------------------

res_sig = res_sig.sort_values("apc", ascending=False)

top2 = res_sig.head(2)
bottom2 = res_sig.tail(2)

res_plot = pd.concat([top2, bottom2])
res_plot = res_plot[~res_plot["Estado"].duplicated()]

# ---------------------------
# 4. plot (2x2 fixo)
# ---------------------------

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, row in enumerate(res_plot.itertuples()):

    ax = axes[i]

    x_vals = anos.flatten()

    ax.plot(x_vals, row.y, marker='o', label='Observado')
    ax.plot(x_vals, row.y_pred, linestyle='--', linewidth=2, label='Prais-Winsten')

    ax.fill_between(
        x_vals,
        row.y_inf,
        row.y_sup,
        alpha=0.1,
        label='IC95%'
    )

    ax.set_title(row.Estado)
    ax.set_xlabel("Ano")
    ax.set_ylabel("Casos")

    texto = (
        f"APC = {row.apc:.2f}%\n"
        f"IC95% = [{row.ic_inf:.2f}; {row.ic_sup:.2f}]\n"
        f"p {format_p(row.p)}"
    )

    ax.text(
        0.05,
        0.95,
        texto,
        transform=ax.transAxes,
        fontsize=8,
        verticalalignment='top',
        bbox=dict(boxstyle="round", alpha=0.2)
    )

    ax.legend(fontsize=7)

# ---------------------------
# remover eixos vazios (caso <4)
# ---------------------------

for j in range(len(res_plot), 4):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

### Mostrando o comportamento de todas as regiões.

In [ ]:
# --- lista de regiões ---
regioes = (
    resultados_df[
        ~resultados_df["Estado"].str.startswith("Região")
    ]["Estado"]
    .tolist()
)

# --- layout ---
n = len(regioes)
ncols = 2
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = axes.reshape(nrows, ncols)

# --- loop ---
for i, regiao in enumerate(regioes):

    row = i // ncols
    col = i % ncols

    ax = axes[row, col]

    y = get_series(regiao)

    # Prais-Winsten
    (
      y_pred,
      y_inf,
      y_sup,
      apc,
      ic_inf,
      ic_sup,
      r2,
      p
    ) = trend_pw(y)

    x_vals = anos.flatten()

    # observados
    ax.plot(
        x_vals,
        y,
        marker='o',
        label='Observado'
    )

    # ajustados
    ax.plot(
        x_vals,
        y_pred,
        linestyle='--',
        linewidth=2,
        label='Prais-Winsten'
    )

    ax.fill_between(
      x_vals,
      y_inf,
      y_sup,
      color="red",
      alpha=0.1,
      label='IC95%'
    )

    ax.set_title(regiao)
    ax.set_xlabel("Ano")
    ax.set_ylabel("Casos")

    texto = (
        f"APC = {apc:.2f}%\n"
        f"IC95% = [{ic_inf:.2f}; {ic_sup:.2f}]\n"
        #f"R² = {r2:.2f}\n"
        f"p {format_p(p)}"
    )

    ax.text(
        0.05,
        0.95,
        texto,
        transform=ax.transAxes,
        fontsize=8,
        verticalalignment='top',
        bbox=dict(
            boxstyle="round",
            alpha=0.2
        )
    )

    ax.legend(fontsize=7)

# --- remover eixos vazios ---
for j in range(i + 1, nrows * ncols):
    fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.show()